### Classificação de dados da CID por capítulo do NTEP

### Definição de Parâmetros 
1. Importação de pacotes e funções 
2. Importação do arquivo dicionário json com cruzamento de CIDs do Decreto do NTEP
3. Definição de função para parsing o CID, definindo capítulos, subcapítulos ou intervalo de CIDs



In [0]:
import json
from pyspark.sql import functions as F

CAMINHO_JSON_NTEP = "/Volumes/workspace/sinan/dados_temp/sinan_ntep_dict.json"
TABELA_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_sinan_consolidado"

with open(CAMINHO_JSON_NTEP, encoding="utf-8") as f:
    ntep_dict = json.load(f)
ntep_dict.pop("_metadata", None)

In [0]:
def parse_padrao_cid(codigo):
    codigo = codigo.strip()
    if "-" in codigo:
        ini, fim = codigo.split("-")
        return {"tipo": "intervalo", "letra": ini[0], "num_min": int(ini[1:3]), "num_max": int(fim[1:3])}
    if "." in codigo:
        return {"tipo": "exato", "codigo": codigo.replace(".", "")}
    return {"tipo": "prefixo", "codigo": codigo}

Orquestração - Aplicação na tabela gerada do JSON
1. Definir se código é do tipo código, capítulo ou intervalo  
2. Definir vetor com equivalência entre Capítulo da CID e letra do código
3. Definir vetor com equivalência entre código numérico e região

In [0]:
padroes = []
for categoria, info in ntep_dict.items():
    for codigo in info["I"]:
        p = parse_padrao_cid(codigo)
        p["categoria_ntep"] = categoria
        p["presente_no_ntep"] = info["presente_no_NTEP"]
        padroes.append(p)

PADROES_EXATO = [p for p in padroes if p["tipo"] == "exato"]
PADROES_PREFIXO = [p for p in padroes if p["tipo"] == "prefixo"]
PADROES_INTERVALO = [p for p in padroes if p["tipo"] == "intervalo"]

MAPA_CAPITULO_LETRA = {
    "A": "Capítulo I", "B": "Capítulo I", "C": "Capítulo II",
    "E": "Capítulo IV", "F": "Capítulo V", "G": "Capítulo VI",
    "I": "Capítulo IX", "J": "Capítulo X", "K": "Capítulo XI",
    "L": "Capítulo XII", "M": "Capítulo XIII", "N": "Capítulo XIV",
    "O": "Capítulo XV", "P": "Capítulo XVI", "Q": "Capítulo XVII",
    "R": "Capítulo XVIII", "S": "Capítulo XIX", "T": "Capítulo XIX",
    "V": "Capítulo XX", "W": "Capítulo XX", "X": "Capítulo XX", "Y": "Capítulo XX",
    "Z": "Capítulo XXI", "U": "Capítulo XXII",
}
MAPA_REGIAO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}

Definir função para cruzar as notificações do SINAN com o dicionário de dados do NTEP

In [0]:
def construir_classificacao(cid_limpo, letra, numero, campo_resultado):
    condicoes = (
        [(cid_limpo == F.lit(p["codigo"]), p[campo_resultado]) for p in PADROES_EXATO]
        + [(F.substring(cid_limpo, 1, 3) == F.lit(p["codigo"]), p[campo_resultado]) for p in PADROES_PREFIXO]
        + [((letra == F.lit(p["letra"])) & numero.between(p["num_min"], p["num_max"]), p[campo_resultado])
           for p in PADROES_INTERVALO]
    )
    expr = F.when(condicoes[0][0], F.lit(condicoes[0][1]))
    for cond, val in condicoes[1:]:
        expr = expr.when(cond, F.lit(val))
    return expr.otherwise(F.lit(None))

Limpeza da tabela para cruzamento e criação de mapas para linguagem spark

In [0]:
df_sinan = spark.table(TABELA_BRONZE)
df_sinan = df_sinan.withColumn("CID_LIMPO", F.upper(F.regexp_replace(F.col("CID10"), r"\.", "")))
cid_limpo = F.col("CID_LIMPO")
letra = F.substring(cid_limpo, 1, 1)
numero = F.expr("try_cast(substring(CID_LIMPO, 2, 2) as int)")

mapa_capitulo = F.create_map([F.lit(x) for kv in MAPA_CAPITULO_LETRA.items() for x in kv])
mapa_regiao = F.create_map([F.lit(x) for kv in MAPA_REGIAO.items() for x in kv])

Gravar a classificação NTEP na nova tabela

In [0]:
df_sinan = (
    df_sinan
    .withColumn("CATEGORIA_NTEP", construir_classificacao(cid_limpo, letra, numero, "categoria_ntep"))
    .withColumn("PRESENTE_NO_NTEP", construir_classificacao(cid_limpo, letra, numero, "presente_no_ntep"))
    .withColumn(
        "CAPITULO_CID_OFICIAL",
        F.when(letra == "D", F.when(numero <= 48, "Capítulo II").otherwise("Capítulo III"))
         .when(letra == "H", F.when(numero <= 59, "Capítulo VII").otherwise("Capítulo VIII"))
         .otherwise(mapa_capitulo[letra])
    )
    .withColumn("REGIAO_NOTIF", mapa_regiao[F.col("COD_REGIAO")])
)

Filtrar apenas a base que está presente no NTEP, e mostrar estatísticas básicas

In [0]:
df_gold_detalhe = df_sinan.filter(F.col("CATEGORIA_NTEP").isNotNull())

total_sinan = df_sinan.count()
total_gold = df_gold_detalhe.count()
print(f"Total bronze: {total_sinan:,}")
print(f"Total com CID de interesse do NTEP: {total_gold:,} ({total_gold / total_sinan * 100:.1f}%)")

Criar bases de notificações agrupadas por categorias de interesse (categoria NTEP, Capítulo CID, Sexo, Raça, com emissão de CAT), e salvar
1. A nível nacional
2. A nível de região
3. A nível de UF

In [0]:
DIMENSOES_BASE = [
    "ANO_NOTIF", "CATEGORIA_NTEP", "CAPITULO_CID_OFICIAL", "PRESENTE_NO_NTEP",
    "SEXO", "RACA", "EMITIA_CAT",
]

gold_brasil = df_gold_detalhe.groupBy(*DIMENSOES_BASE).count().withColumnRenamed("count", "N_NOTIFICACOES")
gold_regiao = df_gold_detalhe.groupBy(*DIMENSOES_BASE, "REGIAO_NOTIF").count().withColumnRenamed("count", "N_NOTIFICACOES")
gold_uf = df_gold_detalhe.groupBy(*DIMENSOES_BASE, "UF_NOTIF").count().withColumnRenamed("count", "N_NOTIFICACOES")

for nome, df in [
    ("gold_sinan_ntep_detalhe", df_gold_detalhe),
    ("gold_sinan_nivel_brasil", gold_brasil),
    ("gold_sinan_nivel_regiao", gold_regiao),
    ("gold_sinan_nivel_uf", gold_uf),
]:
    tabela = f"{CATALOG}.{SCHEMA}.{nome}"
    spark.sql(f"DROP TABLE IF EXISTS {tabela}")
    escritor = df.write.mode("overwrite")
    if nome == "gold_sinan_ntep_detalhe":
        escritor = escritor.partitionBy("CATEGORIA_NTEP")
    escritor.saveAsTable(tabela)
    print(f"{tabela}: {df.count():,} linhas")